# 30 Retos de Ciberseguridad en Python
**Purple Team Series**

Cada ejercicio contiene:
- Descripción del reto
- Pistas para resolverlo
- Celda de código para la solución

---
## Módulo 1 — Criptografía

### Reto 1 — Generador de hashes
**Objetivo:** Escribe una función que reciba un string y devuelva su hash en MD5, SHA1 y SHA256 al mismo tiempo.

**Pistas:**
- Usa el módulo estándar `hashlib`
- `hashlib.md5()`, `hashlib.sha1()`, `hashlib.sha256()`
- Recuerda que debes codificar el string antes: `.encode('utf-8')`
- El método `.hexdigest()` devuelve el hash como string legible
- Devuelve un diccionario con las tres claves: `'md5'`, `'sha1'`, `'sha256'`

In [11]:
import hashlib

def generar_hashes(texto: str) -> dict:
    
    texto_codificado = texto.encode('utf-8')
    hash_md5 = hashlib.md5(texto_codificado).hexdigest()
    hash_sha1 = hashlib.sha1(texto_codificado).hexdigest()
    hash_sha256 = hashlib.sha256(texto_codificado).hexdigest()

    resultado = {'md5': hash_md5, 'sha1': hash_sha1, 'sha256': hash_sha256}

    return resultado

# Prueba
resultado = generar_hashes('anclalab')
print(resultado)

{'md5': '7ec8905e8b846910c8de584e7815c5ed', 'sha1': '67196d5b7b3d239a7561aab6e9ed10aba06fc026', 'sha256': 'd50a18768dde7bcb9e5818b7a818be854a5c537d56cbf654af822328f1a5c044'}


### Reto 2 — Cifrado César
**Objetivo:** Implementa el cifrado César: desplaza cada letra del alfabeto `n` posiciones. Incluye función de cifrado y descifrado.

**Pistas:**
- Usa `ord()` para obtener el valor ASCII de un carácter y `chr()` para convertir de vuelta
- Las letras mayúsculas van de ASCII 65 a 90, minúsculas de 97 a 122
- Usa el operador módulo `%` para que el alfabeto sea circular (después de 'z' vuelve a 'a')
- Los caracteres que no sean letras deben mantenerse sin cambio
- Para descifrar, simplemente desplaza en dirección contraria (`-n`)

In [16]:
def cifrar_cesar(texto: str, n: int) -> str:
    cifrado = ''
    for letra in texto:
        if letra.isupper():
            texto_ascii = ord(letra)
            cifrado += chr((texto_ascii + n - 65) % 26 + 65)
        elif letra.islower():
            texto_ascii = ord(letra)
            cifrado += chr((texto_ascii + n - 97) % 26 + 97)
        else:
            cifrado += letra
    return cifrado

def descifrar_cesar(texto: str, n: int) -> str:
    cifrado = ''
    for letra in texto:
        if letra.isupper():
            texto_ascii = ord(letra)
            cifrado += chr((texto_ascii - n - 65) % 26 + 65)
        elif letra.islower():
            texto_ascii = ord(letra)
            cifrado += chr((texto_ascii - n - 97) % 26 + 97)
        else:
            cifrado += letra
    return cifrado

# Prueba
cifrado = cifrar_cesar('Hola Mundo', 3)
print('Cifrado:', cifrado)
print('Descifrado:', descifrar_cesar(cifrado, 3))

Cifrado: Krod Pxqgr
Descifrado: Hola Mundo


### Reto 3 — Cifrado simétrico AES
**Objetivo:** Cifra y descifra un mensaje usando AES-256 en modo CBC con una clave y un IV aleatorio.

**Pistas:**
- Usa `from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes`
- La clave debe tener exactamente 32 bytes (AES-256)
- El IV debe tener 16 bytes y generarse aleatoriamente con `os.urandom(16)`
- El mensaje debe ser múltiplo de 16 bytes — usa padding: `from cryptography.hazmat.primitives import padding`
- Guarda el IV junto al mensaje cifrado para poder descifrar después

In [ ]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

def cifrar_aes(mensaje: str, clave: bytes) -> bytes:
    # generar el IV
    iv = os.urandom(16)

    # Crea un objeto que sabe cómo aplicar el estándar PKCS7
    padder = padding.PKCS7(128).padder()

    # Convierte el string a bytes
    mensaje_bytes = mensaje.encode('utf-8')

    # Aplica el padding real
    mensaje_padded = padder.update(mensaje_bytes) + padder.finalize()

    # Crea el objeto cifrador. Le dice: usa el algoritmo AES 
    # con esta clave, en modo CBC con este IV.
    cipher = Cipher(algorithms.AES(clave), modes.CBC(iv))
    
    # Del objeto cipher saca el encriptador
    encryptor = cipher.encryptor()
    datos_cifrados = encryptor.update(mensaje_padded) + encryptor.finalize()

    return iv + datos_cifrados

def descifrar_aes(datos: bytes, clave: bytes) -> str:
    iv = datos[:16]
    mensaje_cifrado = datos[16:]
    cipher = Cipher(algorithms.AES(clave), modes.CBC(iv))
    decryptor = cipher.decryptor()
    datos_descifrados = decryptor.update(mensaje_cifrado) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    mensaje = unpadder.update(datos_descifrados) + unpadder.finalize()
    return mensaje.decode('utf-8')

# Prueba
clave = os.urandom(32)
cifrado = cifrar_aes('Mensaje secreto de Ancla Lab', clave)
print('Cifrado (hex):', cifrado.hex())
print('Descifrado:', descifrar_aes(cifrado, clave))

Cifrado (hex): b9a7680dd6494e6d83de78d21ef41ff69b0ac4dffa41a266ab7b1fb57e65cc00a59fe6b38263f4098232849e460fe5cb
Descifrado: Mensaje secreto de Ancla Lab


### Reto 4 — Fuerza bruta a hash MD5
**Objetivo:** Dado un hash MD5, encuentra la contraseña original probando palabras de un diccionario.

**Pistas:**
- Lee un archivo de wordlist línea por línea (puedes crear una lista pequeña de prueba)
- Para cada palabra, calcula su hash MD5 y compara con el objetivo
- Usa `hashlib.md5(palabra.encode()).hexdigest()`
- Recuerda hacer `.strip()` a cada línea para quitar saltos de línea
- Si no encuentra coincidencia, devuelve `None`

In [1]:
import hashlib

# Wordlist de prueba
wordlist = ['password', '123456', 'admin', 'anclalab', 'qwerty', 'letmein']

# Hash objetivo (MD5 de 'anclalab')
hash_objetivo = hashlib.md5('anclalab'.encode()).hexdigest()
print('Hash objetivo:', hash_objetivo)

def fuerza_bruta_md5(hash_obj: str, wordlist: list) -> str:
    for palabra in wordlist:
        palabra_md5 = hashlib.md5(palabra.encode()).hexdigest()
        if palabra_md5 == hash_obj:
            return palabra
    return None

# Prueba
resultado = fuerza_bruta_md5(hash_objetivo, wordlist)
print('Contraseña encontrada:', resultado)

Hash objetivo: 7ec8905e8b846910c8de584e7815c5ed
Contraseña encontrada: anclalab


### Reto 5 — Generador de contraseñas seguras
**Objetivo:** Crea un generador de contraseñas aleatorias que permita configurar longitud e inclusión de mayúsculas, números y símbolos.

**Pistas:**
- Usa el módulo `secrets` (más seguro que `random` para criptografía)
- `import string` tiene constantes útiles: `string.ascii_lowercase`, `string.ascii_uppercase`, `string.digits`, `string.punctuation`
- Construye el alfabeto según los parámetros recibidos
- Usa `secrets.choice(alfabeto)` en un loop
- Asegúrate de que la contraseña tenga al menos un carácter de cada tipo seleccionado

In [6]:
import secrets
import string

def generar_password(longitud=16, mayusculas=True, numeros=True, simbolos=True) -> str:

    contraseña = ''

    alfabeto = string.ascii_lowercase
    if mayusculas:
        alfabeto += string.ascii_uppercase
    if numeros:
        alfabeto += string.digits
    if simbolos:
        alfabeto += string.punctuation
        
    for _ in range(longitud):
        contraseña += secrets.choice(alfabeto)

    return contraseña

# Prueba
print(generar_password(20, mayusculas=True, numeros=True, simbolos=True))
print(generar_password(12, mayusculas=False, numeros=True, simbolos=False))

D0})3B+9%.*j2{8oJ7k]
h3uiu3a2nnsa


---
## Módulo 2 — Redes y Escaneo

### Reto 6 — Port Scanner TCP
**Objetivo:** Escanea un rango de puertos en un host y devuelve los que están abiertos.

**Pistas:**
- Usa el módulo estándar `socket`
- `socket.socket(socket.AF_INET, socket.SOCK_STREAM)` crea un socket TCP
- El método `.connect_ex((host, puerto))` devuelve 0 si el puerto está abierto
- Usa `s.settimeout(0.5)` para no esperar demasiado por puerto
- Cierra el socket después de cada intento con `s.close()`
- Prueba con `localhost` o `127.0.0.1` y puertos 1-1024

In [1]:
import socket

def port_scanner(host: str, puerto_inicio: int, puerto_fin: int) -> list:
    puertos = []
    
    for i in range(puerto_inicio, puerto_fin + 1):
        # Crear socket
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        # Tiempo de espera por puerto
        s.settimeout(0.5)
        # Devuelve 0 si el puerto está abierto
        puerto = s.connect_ex((host, i))
        if puerto == 0:
            puertos.append(i)
        # cerrar socket
        s.close()
    return puertos

# Prueba
host = '127.0.0.1'
puertos_abiertos = port_scanner(host, 1, 1024)
print(f'Puertos abiertos en {host}:', puertos_abiertos)

Puertos abiertos en 127.0.0.1: [135, 445]


### Reto 7 — Banner Grabbing
**Objetivo:** Conéctate a un puerto abierto y captura el banner de servicio que responde el servidor.

**Pistas:**
- Reutiliza `socket` igual que el reto anterior
- Una vez conectado, usa `s.recv(1024)` para recibir datos
- Algunos servicios responden solos (FTP, SMTP); otros necesitan que envíes algo primero
- Para HTTP, envía `b'HEAD / HTTP/1.0\r\n\r\n'` con `s.send()`
- Decodifica la respuesta con `.decode('utf-8', errors='ignore')`
- Usa `try/except` para manejar puertos que no respondan

In [1]:
import socket

def banner_grabbing(host: str, puerto: int, timeout=2) -> str:
    # Crear socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    # Tiempo de espera por puerto
    s.settimeout(timeout)
    # Conectarse al host
    s.connect((host, puerto))
    # Para conecciones http
    s.send(b'HEAD / HTTP/1.0\r\n\r\n')
    # Recibir el número de bytes a leer
    try:
        banner = s.recv(1024)
    except:
        "No response"
    # Cerrar el socket y decodificar
    s.close()
    return banner.decode('utf-8', errors='ignore')

# Prueba con un host público
banner = banner_grabbing('scanme.nmap.org', 80)
print('Banner recibido:')
print(banner)

Banner recibido:
HTTP/1.1 200 OK
Date: Thu, 16 Apr 2026 15:49:13 GMT
Server: Apache/2.4.7 (Ubuntu)
Accept-Ranges: bytes
Vary: Accept-Encoding
Connection: close
Content-Type: text/html




### Reto 8 — Resolución DNS inversa
**Objetivo:** Dada una lista de IPs, obtén el hostname correspondiente a cada una mediante DNS inverso.

**Pistas:**
- Usa `socket.gethostbyaddr(ip)` — devuelve una tupla `(hostname, aliases, addresses)`
- Si la IP no tiene registro PTR, lanza una excepción — manéjala con `try/except socket.herror`
- Devuelve un diccionario `{ip: hostname}` con `'No encontrado'` si falla
- También puedes hacer resolución directa con `socket.gethostbyname(nombre)` para comparar

In [4]:
import socket

def dns_inverso(ips: list) -> dict:
    resultado = {}
    for i in ips:
        try:
            hostname = socket.gethostbyaddr(i)[0]
            resultado[i] = hostname
        except:
            resultado[i] = ['No encontrado']
    return resultado

# Prueba
ips = ['8.8.8.8', '1.1.1.1', '192.168.1.1', '45.33.32.156', '37.19.199.154']
resultado = dns_inverso(ips)
for ip, host in resultado.items():
    print(f'{ip} -> {host}')

8.8.8.8 -> dns.google
1.1.1.1 -> one.one.one.one
192.168.1.1 -> ['No encontrado']
45.33.32.156 -> scanme.nmap.org
37.19.199.154 -> unn-37-19-199-154.datapacket.com


### Reto 9 — Geolocalización de IP
**Objetivo:** Consulta la API pública de `ip-api.com` para obtener país, ciudad, ISP y coordenadas de una IP.

**Pistas:**
- Usa `requests.get(url)` con la URL: `http://ip-api.com/json/{ip}`
- La respuesta es JSON — usa `.json()` para convertirla
- Campos útiles: `country`, `city`, `isp`, `lat`, `lon`, `org`
- Si `status == 'fail'`, la IP es privada o inválida
- No abuses de la API — tiene límite de 45 peticiones por minuto sin API key

In [2]:
import requests

def geolocalizacion_ip(ip: str) -> dict:
    url = f'http://ip-api.com/json/{ip}'
    api_request = requests.get(url)
    datos = api_request.json()
    if datos['status'] == 'fail':
        return "La IP es privada o invalida"
    return datos

# Prueba
info = geolocalizacion_ip('8.8.8.8')
for clave, valor in info.items():
    print(f'{clave}: {valor}')

status: success
country: United States
countryCode: US
region: VA
regionName: Virginia
city: Ashburn
zip: 20149
lat: 39.03
lon: -77.5
timezone: America/New_York
isp: Google LLC
org: Google Public DNS
as: AS15169 Google LLC
query: 8.8.8.8


### Reto 10 — Sniffer de paquetes básico
**Objetivo:** Captura paquetes de red en tiempo real y muestra IP origen, IP destino y protocolo.

**Pistas:**
- Usa `scapy`: `from scapy.all import sniff, IP, TCP, UDP`
- La función `sniff(count=N, prn=callback)` captura N paquetes y ejecuta el callback por cada uno
- En el callback recibe el paquete como argumento: `def procesar(pkt)`
- Verifica si tiene capa IP con `if IP in pkt`
- Accede a campos con `pkt[IP].src`, `pkt[IP].dst`, `pkt[IP].proto`
- Proto 6 = TCP, 17 = UDP, 1 = ICMP

> Requiere permisos de administrador en Windows. Ejecuta VSCode como admin si es necesario.

In [3]:
from scapy.all import sniff, IP, TCP, UDP

def procesar_paquete(pkt):
    if IP in pkt:
        lista = [pkt[IP].src, pkt[IP].dst, pkt[IP].proto]
        print(lista)

print('Capturando 10 paquetes...')
sniff(count=10, prn=procesar_paquete)

Capturando 10 paquetes...
['192.168.1.91', '37.19.199.154', 17]
['192.168.1.91', '37.19.199.154', 17]
['37.19.199.154', '192.168.1.91', 17]
['37.19.199.154', '192.168.1.91', 17]
['37.19.199.154', '192.168.1.91', 17]
['37.19.199.154', '192.168.1.91', 17]
['192.168.1.91', '37.19.199.154', 17]
['192.168.1.91', '37.19.199.154', 17]
['192.168.1.164', '192.168.1.255', 17]


<Sniffed: TCP:0 UDP:9 ICMP:0 Other:1>

---
## Módulo 3 — Análisis Web

### Reto 11 — Analizador de headers HTTP
**Objetivo:** Dado un dominio, analiza sus headers de seguridad HTTP e indica cuáles faltan o están mal configurados.

**Pistas:**
- Usa `requests.get(url)` y accede a `response.headers`
- Headers importantes a verificar: `Strict-Transport-Security`, `Content-Security-Policy`, `X-Frame-Options`, `X-Content-Type-Options`, `Referrer-Policy`, `Permissions-Policy`
- Devuelve un reporte con qué headers están presentes y cuáles faltan
- Usa `response.headers.get('nombre', None)` para verificar cada uno

In [5]:
import requests

HEADERS_SEGURIDAD = [
    'Strict-Transport-Security',
    'Content-Security-Policy',
    'X-Frame-Options',
    'X-Content-Type-Options',
    'Referrer-Policy',
    'Permissions-Policy'
]

def analizar_headers(url: str) -> dict:
    obtener_url = requests.get(url)
    resultado = {}
    for nombre in HEADERS_SEGURIDAD:
        respuesta_headers = obtener_url.headers.get(nombre, None)
        resultado[nombre] = respuesta_headers
    return resultado

# Prueba
reporte = analizar_headers('https://github.com')
for header, valor in reporte.items():
    estado = '✅' if valor else '❌'
    print(f'{estado} {header}: {valor or "AUSENTE"}')

✅ Strict-Transport-Security: max-age=31536000; includeSubdomains; preload
✅ Content-Security-Policy: default-src 'none'; base-uri 'self'; child-src github.githubassets.com github.com/assets-cdn/worker/ github.com/assets/ gist.github.com/assets-cdn/worker/; connect-src 'self' uploads.github.com www.githubstatus.com collector.github.com raw.githubusercontent.com api.github.com github-cloud.s3.amazonaws.com github-production-repository-file-5c1aeb.s3.amazonaws.com github-production-upload-manifest-file-7fdce7.s3.amazonaws.com github-production-user-asset-6210df.s3.amazonaws.com *.rel.tunnels.api.visualstudio.com wss://*.rel.tunnels.api.visualstudio.com github.githubassets.com objects-origin.githubusercontent.com copilot-proxy.githubusercontent.com proxy.individual.githubcopilot.com proxy.business.githubcopilot.com proxy.enterprise.githubcopilot.com *.actions.githubusercontent.com wss://*.actions.githubusercontent.com productionresultssa0.blob.core.windows.net productionresultssa1.blob.cor

### Reto 12 — Detector de SQLi básico
**Objetivo:** Prueba parámetros GET de una URL con payloads de SQL Injection y detecta posibles vulnerabilidades por errores en la respuesta.

**Pistas:**
- Usa `requests.get(url, params={'param': payload})`
- Payloads básicos: `"'"`, `"' OR '1'='1"`, `"'; DROP TABLE--"`, `"1 AND 1=1"`, `"1 AND 1=2"`
- Busca en la respuesta errores SQL típicos: `'sql'`, `'syntax'`, `'mysql'`, `'ORA-'`, `'error'`
- Usa `response.text.lower()` para buscar sin importar mayúsculas
- Prueba en DVWA con `security=low` en tu lab local

> Solo usar en entornos propios.

In [ ]:
import requests

PAYLOADS_SQLI = ["'", "' OR '1'='1", "'; DROP TABLE--", "1 AND 1=1", "1 AND 1=2"]
ERRORES_SQL = ['sql', 'syntax', 'mysql', 'ora-', 'error in your sql', 'sqlite']

def detectar_sqli(url: str, parametro: str, cookies: dict = None) -> list:
    lista = []
    for payload in PAYLOADS_SQLI:
        response = requests.get(url, params={parametro: payload}, cookies=cookies)
        html_error = response.text.lower()
        for error in ERRORES_SQL:
            if error in html_error:
                lista.append(payload)
                break
    return lista

# Prueba con DVWA local (ajusta la URL y cookies según tu sesión)
# url = ''
# cookies = {'PHPSESSID': 'TU_SESION', 'security': 'low'}
resultados = detectar_sqli(url, 'id')
print(resultados)

[]


### Reto 13 — Escáner de XSS Reflejado
**Objetivo:** Prueba si un parámetro GET es vulnerable a XSS reflejado verificando si el payload aparece sin sanitizar en la respuesta.

**Pistas:**
- El principio es simple: si envías `<script>alert(1)</script>` y aparece igual en el HTML de respuesta, hay XSS
- Usa `payload in response.text` para verificar el reflejo
- Prueba varios payloads: `<script>alert(1)</script>`, `<img src=x onerror=alert(1)>`, `"><script>alert(1)</script>`
- Distingue entre reflejo sin encoding (vulnerable) y con encoding como `&lt;script&gt;` (no vulnerable)
- Busca también en atributos HTML dentro de la respuesta

In [21]:
import requests

PAYLOADS_XSS = [
    '<script>alert(1)</script>',
    '<img src=x onerror=alert(1)>',
    '"><script>alert(1)</script>',
    "'><script>alert(1)</script>"
]

def detectar_xss(url: str, parametro: str, cookies: dict = None) -> list:
    lista = []
    for payload in PAYLOADS_XSS:
        response = requests.get(url, params={parametro: payload}, cookies=cookies)
        if payload in response.text:
            lista.append(f'{payload}, Vulnerable — el payload se refleja sin sanitizar')
    return lista

# Prueba con DVWA
url = 'http://192.168.1.96:8080/vulnerabilities/xss_r/'
cookies = {'PHPSESSID': 'lht2k0kug434f667eckvcui5h1', 'security': 'low'}
print(detectar_xss(url, 'name', cookies))

['<script>alert(1)</script>, Vulnerable — el payload se refleja sin sanitizar', '<img src=x onerror=alert(1)>, Vulnerable — el payload se refleja sin sanitizar', '"><script>alert(1)</script>, Vulnerable — el payload se refleja sin sanitizar', "'><script>alert(1)</script>, Vulnerable — el payload se refleja sin sanitizar"]


### Reto 14 — Directory Brute-Force
**Objetivo:** Descubre rutas y directorios ocultos en un servidor web probando palabras de una wordlist.

**Pistas:**
- Construye la URL completa: `url_base + '/' + palabra`
- Usa `requests.get(url, allow_redirects=False)` 
- Considera positivos los códigos 200, 301, 302, 403
- Ignora 404 — esos no existen
- Usa `timeout=3` para no quedarte colgado
- Wordlist mínima: `['admin', 'login', 'backup', 'config', 'api', 'uploads', 'images', 'test', 'dev', 'phpmyadmin']`

In [3]:
import requests

wordlist = ['admin.php', 'login.php', 'setup.php', 'config.php', 
            'phpinfo.php', 'uploads', 'images', 'phpmyadmin',
            'vulnerabilities', 'security.php', 'logout.php']

def directory_bruteforce(url_base: str, wordlist: list, cookies: dict = None) -> list:
    lista = []
    for palabra in wordlist:
        url_completa = url_base + '/' + palabra
        response = requests.get(url_completa, allow_redirects=False, timeout=3, cookies=cookies)
        if response.status_code in [200, 301, 302, 403]:
            lista.append(f'{url_completa} - {response.status_code}')
    return lista

# Prueba con DVWA
cookies = {'PHPSESSID': 's43eihvhvj3dmtb05m4v8ll7q0', 'security': 'low'}
resultados = directory_bruteforce('http://192.168.1.96:8080', wordlist, cookies=cookies)
for r in resultados:
    print(r)

http://192.168.1.96:8080/login.php - 200
http://192.168.1.96:8080/setup.php - 200
http://192.168.1.96:8080/phpinfo.php - 302
http://192.168.1.96:8080/vulnerabilities - 301
http://192.168.1.96:8080/security.php - 302
http://192.168.1.96:8080/logout.php - 302


### Reto 15 — Extractor de links y formularios
**Objetivo:** Dado un URL, extrae todos los links (`<a href>`) y formularios (`<form>`) con sus campos y métodos.

**Pistas:**
- Usa `requests` para obtener el HTML y `from html.parser import HTMLParser` para parsearlo
- O más fácil: instala `pip install beautifulsoup4` y usa `BeautifulSoup(html, 'html.parser')`
- Links: `soup.find_all('a', href=True)` → cada elemento tiene `['href']`
- Formularios: `soup.find_all('form')` → atributos `action`, `method`
- Campos dentro de cada form: `form.find_all(['input', 'textarea', 'select'])`
- Devuelve un dict con claves `'links'` y `'formularios'`

In [1]:
import requests
from bs4 import BeautifulSoup  # pip install beautifulsoup4

def extraer_links_formularios(url: str, cookies: dict = None) -> dict:
    resultado = {'links': [], 'formularios': []}
    response = requests.get(url, cookies=cookies)
    soup = BeautifulSoup(response.text, 'html.parser')
    links = soup.find_all('a', href=True)
    forms = soup.find_all('form')

    for link in links:
        url_link = link['href']
        resultado['links'].append(url_link)

    for form in forms:
        formulario = {
        'action': form.get('action', ''),
        'method': form.get('method', 'get'),
        'campos': []
        }  
        # Iterar campos dentro de cada formulario
        for campo in form.find_all(['input', 'textarea', 'select']):
            formulario['campos'].append(campo.get('name', ''))
        resultado['formularios'].append(formulario)
    return resultado

# Prueba
cookies = {'PHPSESSID': '7015depn4jarg4jgtvj0a2o215', 'security': 'low'}
datos = extraer_links_formularios('http://192.168.1.96:8080/login.php', cookies=cookies)
print('Links encontrados:', len(datos['links']))
print('Formularios encontrados:', len(datos['formularios']))
for form in datos['formularios']:
    print(form)

Links encontrados: 1
Formularios encontrados: 1
{'action': 'login.php', 'method': 'post', 'campos': ['username', 'password', 'Login', 'user_token']}


---
## Módulo 4 — Forense Digital

### Reto 16 — Hash de integridad de archivos
**Objetivo:** Calcula el hash SHA256 de un archivo y permite verificar si fue modificado comparando con un hash de referencia.

**Pistas:**
- Abre el archivo en modo binario: `open(ruta, 'rb')`
- Lee en bloques para no saturar memoria: `while chunk := f.read(8192)`
- Usa `hashlib.sha256()` y actualiza con cada bloque: `h.update(chunk)`
- Al final llama `h.hexdigest()` para obtener el hash final
- La función de verificación simplemente compara el hash calculado con el esperado

In [ ]:
import hashlib

def hash_archivo(ruta: str) -> str:
    # Tu código aquí
    pass

def verificar_integridad(ruta: str, hash_esperado: str) -> bool:
    # Tu código aquí
    pass

# Prueba: crea un archivo de prueba
with open('prueba.txt', 'w') as f:
    f.write('Archivo de prueba Ancla Lab')

hash_original = hash_archivo('prueba.txt')
print('Hash SHA256:', hash_original)
print('Verificación:', verificar_integridad('prueba.txt', hash_original))

### Reto 17 — Extractor de metadatos de imágenes (EXIF)
**Objetivo:** Extrae los metadatos EXIF de una imagen JPG, incluyendo GPS si está disponible.

**Pistas:**
- Instala: `pip install Pillow`
- Usa `from PIL import Image` y `from PIL.ExifTags import TAGS`
- `img._getexif()` devuelve un diccionario con tags numéricos
- Convierte los tags numéricos a nombres con `TAGS.get(tag_id, tag_id)`
- Para GPS: el tag se llama `GPSInfo` y contiene subdatos — busca latitud y longitud
- Si no tiene EXIF, `_getexif()` devuelve `None`

In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS

def extraer_exif(ruta_imagen: str) -> dict:
    # Tu código aquí
    pass

# Prueba con cualquier JPG que tengas
# metadata = extraer_exif('foto.jpg')
# for clave, valor in metadata.items():
#     print(f'{clave}: {valor}')

### Reto 18 — Analizador de logs de acceso Apache/Nginx
**Objetivo:** Parsea un archivo de log de acceso web y genera estadísticas: IPs más frecuentes, códigos de respuesta, rutas más solicitadas.

**Pistas:**
- El formato Common Log es: `IP - - [fecha] "MÉTODO ruta protocolo" código bytes`
- Usa `re.match()` con una expresión regular para extraer los campos
- Patrón útil: `r'(\S+) \S+ \S+ \[.*?\] "(\S+) (\S+) \S+" (\d+) (\S+)'`
- Usa `collections.Counter` para contar frecuencias fácilmente
- `.most_common(10)` devuelve los 10 más frecuentes

In [ ]:
import re
from collections import Counter

# Log de prueba
log_prueba = """192.168.1.10 - - [13/Apr/2026:10:00:01 +0000] "GET /index.php HTTP/1.1" 200 512
192.168.1.20 - - [13/Apr/2026:10:00:02 +0000] "POST /login.php HTTP/1.1" 302 0
10.0.0.5 - - [13/Apr/2026:10:00:03 +0000] "GET /admin HTTP/1.1" 403 256
192.168.1.10 - - [13/Apr/2026:10:00:04 +0000] "GET /index.php HTTP/1.1" 200 512
192.168.1.99 - - [13/Apr/2026:10:00:05 +0000] "GET /shell.php HTTP/1.1" 404 0
192.168.1.10 - - [13/Apr/2026:10:00:06 +0000] "GET /images/logo.png HTTP/1.1" 200 1024"""

def analizar_log(contenido_log: str) -> dict:
    # Tu código aquí
    pass

estadisticas = analizar_log(log_prueba)
print('Top IPs:', estadisticas.get('top_ips'))
print('Códigos:', estadisticas.get('codigos'))
print('Top rutas:', estadisticas.get('top_rutas'))

### Reto 19 — Buscador de archivos sensibles
**Objetivo:** Recorre un directorio recursivamente y encuentra archivos que puedan contener información sensible por su extensión o nombre.

**Pistas:**
- Usa `os.walk(directorio)` para recorrer recursivamente
- Extensiones sospechosas: `.env`, `.bak`, `.sql`, `.log`, `.key`, `.pem`, `.cfg`, `.conf`
- Nombres sospechosos: `password`, `secret`, `credentials`, `token`, `backup`
- Compara con `archivo.endswith(ext)` y `nombre in archivo.lower()`
- Devuelve lista de rutas completas con `os.path.join(raiz, archivo)`

In [ ]:
import os

EXTENSIONES_SENSIBLES = ['.env', '.bak', '.sql', '.log', '.key', '.pem', '.cfg', '.conf', '.ini']
NOMBRES_SENSIBLES = ['password', 'secret', 'credentials', 'token', 'backup', 'private']

def buscar_archivos_sensibles(directorio: str) -> list:
    # Tu código aquí
    pass

# Prueba con el directorio actual
encontrados = buscar_archivos_sensibles('.')
print(f'Archivos sensibles encontrados: {len(encontrados)}')
for f in encontrados:
    print(' -', f)

### Reto 20 — Detector de strings sospechosos en binarios
**Objetivo:** Abre un archivo binario y extrae todas las cadenas de texto ASCII legibles (como hace el comando `strings` en Linux).

**Pistas:**
- Abre el archivo en modo `'rb'` (binario)
- Recorre byte a byte buscando secuencias de caracteres ASCII imprimibles
- Un carácter es ASCII imprimible si `32 <= byte <= 126`
- Acumula los caracteres hasta encontrar uno no imprimible, y guarda la cadena si tiene longitud mínima (ej. 4 caracteres)
- Busca en el resultado strings como `http://`, `password`, `cmd`, `eval`

In [ ]:
def extraer_strings(ruta: str, longitud_minima: int = 4) -> list:
    # Tu código aquí
    pass

def buscar_iocs(strings: list) -> list:
    """Busca Indicators of Compromise en la lista de strings"""
    iocs = ['http://', 'https://', 'password', 'cmd', 'eval', 'exec', '/bin/sh', 'base64']
    # Tu código aquí
    pass

# Prueba con cualquier ejecutable
# strings = extraer_strings('C:/Windows/System32/notepad.exe')
# print(f'Strings encontrados: {len(strings)}')
# print('IoCs:', buscar_iocs(strings))

---
## Módulo 5 — Exploits y Post-Explotación

### Reto 21 — Encoder de payloads en Base64
**Objetivo:** Codifica y decodifica payloads en Base64 y URL-encoding para evadir filtros simples.

**Pistas:**
- Para Base64: `import base64` → `base64.b64encode(datos.encode()).decode()`
- Para URL encoding: `from urllib.parse import quote, unquote`
- `quote('texto')` convierte caracteres especiales a `%XX`
- Para doble encoding, aplica `quote()` dos veces
- Implementa también la función inversa de decodificación para cada tipo

In [ ]:
import base64
from urllib.parse import quote, unquote

def encode_base64(payload: str) -> str:
    # Tu código aquí
    pass

def decode_base64(payload: str) -> str:
    # Tu código aquí
    pass

def encode_url(payload: str, doble: bool = False) -> str:
    # Tu código aquí
    pass

# Prueba
payload = '<script>alert("XSS")</script>'
print('Original:', payload)
print('Base64:', encode_base64(payload))
print('URL encoded:', encode_url(payload))
print('Doble URL:', encode_url(payload, doble=True))

### Reto 22 — Fuzzer de parámetros HTTP
**Objetivo:** Envía múltiples variaciones de un parámetro a un endpoint y registra respuestas anómalas (cambios en tamaño, código o tiempo de respuesta).

**Pistas:**
- Genera variaciones: strings muy largos, caracteres especiales, números negativos, valores nulos
- Mide el tiempo de respuesta con `time.time()` antes y después de cada request
- Compara el `len(response.text)` entre requests — diferencias grandes son interesantes
- Registra código HTTP, tiempo y tamaño por cada payload
- Un timeout mayor a 5s podría indicar SQLi basado en tiempo

In [ ]:
import requests
import time

PAYLOADS_FUZZ = [
    '', 'A' * 1000, '\x00', '../../../etc/passwd',
    "' OR 1=1--", '<script>alert(1)</script>',
    '%00', '{{7*7}}', '${7*7}', 'null', 'undefined',
    '-1', '0', '9999999'
]

def fuzzer(url: str, parametro: str, cookies: dict = None) -> list:
    # Tu código aquí
    pass

# Prueba con DVWA
# resultados = fuzzer('http://192.168.1.96/dvwa/vulnerabilities/sqli/', 'id',
#                     cookies={'PHPSESSID': 'TU_SESION', 'security': 'low'})
# for r in resultados:
#     print(r)

### Reto 23 — Bind Shell básico
**Objetivo:** Implementa un servidor que escucha en un puerto y ejecuta comandos recibidos, devolviendo el output. Implementa también el cliente.

**Pistas:**
- Servidor: `socket.bind()` → `socket.listen()` → `socket.accept()`
- Recibe el comando con `conn.recv(1024).decode()`
- Ejecuta con `subprocess.run(cmd, shell=True, capture_output=True)`
- Devuelve `stdout` + `stderr` combinados
- Cliente: conecta al servidor, envía comandos y muestra respuestas en loop

> ⚠️ Solo en tu red de lab. Nunca en producción ni redes externas.

In [ ]:
import socket
import subprocess
import threading

def servidor_bind(host='127.0.0.1', puerto=4444):
    """Ejecuta en un thread separado"""
    # Tu código aquí
    pass

def cliente_bind(host='127.0.0.1', puerto=4444):
    # Tu código aquí
    pass

# Para probar, ejecuta servidor en un thread y cliente en otro
# t = threading.Thread(target=servidor_bind, daemon=True)
# t.start()
# import time; time.sleep(0.5)
# cliente_bind()

### Reto 24 — Detector de LFI (Local File Inclusion)
**Objetivo:** Prueba si un parámetro de URL es vulnerable a LFI intentando leer archivos del sistema como `/etc/passwd` o `C:/Windows/win.ini`.

**Pistas:**
- Payloads de path traversal: `../../../etc/passwd`, `....//....//etc/passwd`, `%2e%2e%2fetc%2fpasswd`
- En Windows: `..\..\..\Windows\win.ini`, `C:\Windows\win.ini`
- Indica éxito si la respuesta contiene `'root:'` (Linux) o `'[extensions]'` (Windows)
- También busca `'www-data'`, `'daemon'`, `'nobody'` que son usuarios típicos de `/etc/passwd`
- Prueba con DVWA → File Inclusion

In [ ]:
import requests

PAYLOADS_LFI = [
    '../../../etc/passwd',
    '....//....//....//etc/passwd',
    '%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2fpasswd',
    '../../../Windows/win.ini',
    'C:\\Windows\\win.ini'
]

FIRMAS_EXITO = ['root:', 'www-data', 'daemon', '[extensions]', '[fonts]']

def detectar_lfi(url: str, parametro: str, cookies: dict = None) -> list:
    # Tu código aquí
    pass

# Prueba con DVWA
# url = 'http://192.168.1.96/dvwa/vulnerabilities/fi/'
# cookies = {'PHPSESSID': 'TU_SESION', 'security': 'low'}
# print(detectar_lfi(url, 'page', cookies))

### Reto 25 — Keylogger básico
**Objetivo:** Captura las teclas presionadas durante un tiempo determinado y guarda el registro en un archivo.

**Pistas:**
- Instala: `pip install pynput`
- `from pynput.keyboard import Key, Listener`
- Define `on_press(key)` y `on_release(key)` como callbacks
- En `on_press`: guarda `str(key)` en una lista o archivo
- Teclas especiales como Enter, Space tienen representación especial: `Key.enter`, `Key.space`
- Usa `Listener(on_press=on_press)` y `.start()` / `.stop()`
- Detén la captura tras N segundos con `threading.Timer`

> ⚠️ Solo en tu propia máquina con fines educativos.

In [ ]:
from pynput.keyboard import Key, Listener
import threading

teclas_capturadas = []

def on_press(key):
    # Tu código aquí
    pass

def iniciar_keylogger(duracion_seg: int = 10, archivo_salida: str = 'keylog.txt'):
    # Tu código aquí
    pass

# Prueba — captura 10 segundos
# print('Iniciando captura por 10 segundos...')
# iniciar_keylogger(10)

---
## Módulo 6 — Blue Team / Detección

### Reto 26 — Detector de escaneo de puertos
**Objetivo:** Analiza un archivo de log o tráfico simulado y detecta patrones de port scanning (muchas conexiones desde la misma IP a diferentes puertos en poco tiempo).

**Pistas:**
- Define un umbral: si una IP toca más de N puertos distintos en M segundos, es un scan
- Usa un diccionario `{ip: set(puertos)}` para contar puertos únicos por IP
- Puedes simular el input como lista de tuplas `(timestamp, ip_origen, puerto_destino)`
- Ordena los eventos por timestamp para analizar ventanas de tiempo
- Un threshold razonable: 10 puertos distintos en 5 segundos

In [ ]:
from collections import defaultdict
import time

# Datos simulados: (timestamp, ip_origen, puerto_destino)
eventos_red = [
    (1000.0, '192.168.1.50', 22),
    (1000.1, '192.168.1.50', 80),
    (1000.2, '192.168.1.50', 443),
    (1000.3, '192.168.1.50', 8080),
    (1000.4, '192.168.1.50', 3306),
    (1000.5, '192.168.1.50', 21),
    (1000.6, '192.168.1.50', 25),
    (1000.7, '192.168.1.50', 53),
    (1000.8, '192.168.1.50', 110),
    (1000.9, '192.168.1.50', 143),
    (1000.9, '192.168.1.50', 445),  # 11 puertos < 1 segundo = scan
    (1001.0, '10.0.0.5', 80),       # IP legítima, solo un puerto
]

def detectar_port_scan(eventos: list, umbral_puertos: int = 10, ventana_seg: float = 5.0) -> list:
    # Tu código aquí
    pass

alertas = detectar_port_scan(eventos_red)
for alerta in alertas:
    print('🚨 ALERTA:', alerta)

### Reto 27 — Parser de alertas Wazuh
**Objetivo:** Lee un archivo de alertas JSON de Wazuh y filtra por nivel de severidad, regla o agente, generando un resumen.

**Pistas:**
- Las alertas de Wazuh son JSON, una por línea (formato JSONL)
- Campos clave: `rule.level`, `rule.description`, `agent.name`, `timestamp`
- Usa `json.loads(linea)` para parsear cada línea
- Filtra alertas con nivel >= umbral (ej. nivel >= 7 es alto)
- Usa `collections.Counter` para las reglas más frecuentes
- Maneja excepciones por líneas malformadas con `try/except`

In [ ]:
import json
from collections import Counter

# Alertas simuladas de Wazuh
alertas_json = '''{
  "timestamp": "2026-04-13T10:00:00",
  "rule": {"level": 10, "description": "Multiple authentication failures", "id": "5710"},
  "agent": {"name": "Windows-Marco", "ip": "192.168.1.100"}
}
{"timestamp": "2026-04-13T10:01:00", "rule": {"level": 5, "description": "Log cleared", "id": "1002"}, "agent": {"name": "Windows-Marco", "ip": "192.168.1.100"}}
{"timestamp": "2026-04-13T10:02:00", "rule": {"level": 12, "description": "Rootkit detection", "id": "510"}, "agent": {"name": "Ubuntu-Lab", "ip": "192.168.1.96"}}
{"timestamp": "2026-04-13T10:03:00", "rule": {"level": 3, "description": "New dpkg installed", "id": "2900"}, "agent": {"name": "Ubuntu-Lab", "ip": "192.168.1.96"}}'''

def parsear_alertas_wazuh(contenido: str, nivel_minimo: int = 7) -> dict:
    # Tu código aquí
    pass

reporte = parsear_alertas_wazuh(alertas_json, nivel_minimo=7)
print('Alertas críticas:', reporte.get('total_criticas'))
print('Reglas más frecuentes:', reporte.get('top_reglas'))
print('Agentes afectados:', reporte.get('agentes'))

### Reto 28 — Analizador de PCAP
**Objetivo:** Lee un archivo `.pcap` y extrae estadísticas: conversaciones TCP, peticiones DNS, peticiones HTTP y posibles credenciales en texto claro.

**Pistas:**
- Usa `from scapy.all import rdpcap, IP, TCP, UDP, DNS, Raw`
- `rdpcap('archivo.pcap')` carga todos los paquetes
- Para DNS: `if DNS in pkt and pkt[DNS].qr == 0` → es una query
- Para HTTP: busca paquetes TCP al puerto 80 con `Raw` layer — `pkt[Raw].load.decode(errors='ignore')`
- Busca `'Authorization:'`, `'password='`, `'pass='` en el payload crudo
- Puedes crear un pcap de prueba con Wireshark en tu lab

In [ ]:
from scapy.all import rdpcap, IP, TCP, UDP, DNS, Raw
from collections import Counter

def analizar_pcap(ruta_pcap: str) -> dict:
    # Tu código aquí
    pass

# Prueba con un pcap de tu lab
# resultado = analizar_pcap('captura.pcap')
# print('Conversaciones TCP:', resultado.get('tcp_conversations'))
# print('Queries DNS:', resultado.get('dns_queries'))
# print('Credenciales detectadas:', resultado.get('credenciales'))

### Reto 29 — Correlacionador de eventos de seguridad
**Objetivo:** Dado un conjunto de eventos de distintas fuentes (firewall, IDS, auth logs), correlaciónalos por IP y tiempo para identificar ataques multi-etapa.

**Pistas:**
- Normaliza los eventos a un formato común: `{timestamp, ip, fuente, tipo, descripcion}`
- Agrupa por IP usando `defaultdict(list)`
- Define reglas de correlación: ej. si una IP tiene scan + auth_failure + exploit en < 60 seg → ataque
- Ordena los eventos de cada IP por timestamp
- Asigna un score de riesgo según los tipos de eventos encontrados

In [ ]:
from collections import defaultdict

# Eventos simulados de múltiples fuentes
eventos = [
    {'ts': 1000, 'ip': '10.0.0.99', 'fuente': 'firewall', 'tipo': 'port_scan', 'desc': 'Scan detectado'},
    {'ts': 1015, 'ip': '10.0.0.99', 'fuente': 'auth_log', 'tipo': 'auth_failure', 'desc': 'SSH brute force'},
    {'ts': 1030, 'ip': '10.0.0.99', 'fuente': 'ids', 'tipo': 'exploit', 'desc': 'Shellcode detectado'},
    {'ts': 1045, 'ip': '10.0.0.99', 'fuente': 'ids', 'tipo': 'c2_beacon', 'desc': 'Beaconing C2'},
    {'ts': 2000, 'ip': '192.168.1.5', 'fuente': 'auth_log', 'tipo': 'auth_failure', 'desc': 'Login fallido'},
    {'ts': 2300, 'ip': '192.168.1.5', 'fuente': 'auth_log', 'tipo': 'auth_success', 'desc': 'Login exitoso'},
]

SCORES = {'port_scan': 2, 'auth_failure': 3, 'exploit': 8, 'c2_beacon': 10, 'auth_success': 1}

def correlacionar_eventos(eventos: list, ventana_seg: int = 120) -> list:
    # Tu código aquí
    pass

amenazas = correlacionar_eventos(eventos)
for a in amenazas:
    print('🚨', a)

### Reto 30 — Generador de reporte de incidente
**Objetivo:** Integra los resultados de los módulos anteriores y genera un reporte de incidente en formato Markdown con hallazgos, IOCs, línea de tiempo y recomendaciones.

**Pistas:**
- Recibe como entrada: lista de alertas, lista de IOCs (IPs, hashes, dominios), eventos correlacionados
- Estructura del reporte: Resumen ejecutivo → Línea de tiempo → IOCs → Hallazgos técnicos → Recomendaciones
- Ordena los eventos por timestamp para construir la línea de tiempo
- Usa f-strings para construir el Markdown
- Guarda el resultado en `reporte_incidente.md` con `open('reporte_incidente.md', 'w')`
- Incluye fecha/hora del reporte con `datetime.now()`

In [ ]:
from datetime import datetime

def generar_reporte(alertas: list, iocs: dict, eventos_correlacionados: list,
                    analista: str = 'Ancla Lab') -> str:
    # Tu código aquí
    # Debe devolver el reporte como string Markdown
    pass

# Datos de prueba
alertas_prueba = [
    {'timestamp': '2026-04-13T10:00', 'nivel': 10, 'descripcion': 'Multiple auth failures', 'agente': 'Windows-Marco'},
    {'timestamp': '2026-04-13T10:02', 'nivel': 12, 'descripcion': 'Rootkit detection', 'agente': 'Ubuntu-Lab'},
]
iocs_prueba = {
    'ips': ['10.0.0.99', '192.168.1.50'],
    'hashes': ['d41d8cd98f00b204e9800998ecf8427e'],
    'dominios': ['malware-c2.example.com']
}
eventos_prueba = [
    {'ip': '10.0.0.99', 'score': 23, 'tipos': ['port_scan', 'auth_failure', 'exploit', 'c2_beacon']}
]

reporte = generar_reporte(alertas_prueba, iocs_prueba, eventos_prueba)
if reporte:
    with open('reporte_incidente.md', 'w', encoding='utf-8') as f:
        f.write(reporte)
    print('✅ Reporte generado: reporte_incidente.md')
    print(reporte[:500] + '...')